# Venture Capital Funding Prediction

## Introduction
This notebook builds a regression model to predict total startup funding. All bugs from the original version have been fixed.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Load Dataset

In [ ]:
df = pd.read_csv("investments_VC.csv", encoding='latin-1')

# FIX 1: Strip whitespace from column names (columns like ' funding_total_usd ' had spaces)
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
df.head()

## 3. Data Understanding

In [ ]:
df.info()
df.isnull().sum()

## 4. Data Cleaning

> **Bug Fix 1:** `funding_total_usd` must be cleaned and converted to numeric BEFORE calling `dropna`. In the original, `dropna` was called on a string column so no rows were actually dropped (NaN vs string '-').

In [ ]:
# FIX 2: Convert funding column to numeric BEFORE dropna
df['funding_total_usd'] = df['funding_total_usd'].astype(str).str.replace(",", "").str.strip()
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

df = df.dropna(subset=['funding_total_usd'])
df = df[df['funding_total_usd'] > 0]   # must be positive for log transform later

df.fillna(df.median(numeric_only=True), inplace=True)
df.drop_duplicates(inplace=True)

print("Shape after cleaning:", df.shape)

## 5. Feature Engineering

> **Bug Fix 2:** `funding_gap` was a Timedelta object and was never converted to a number. It caused errors in outlier detection and model training. Fixed by extracting `.dt.days`.

In [ ]:
# FIX 3: Convert timedelta to numeric (days)
df['first_funding_at'] = pd.to_datetime(df['first_funding_at'], errors='coerce')
df['last_funding_at']  = pd.to_datetime(df['last_funding_at'],  errors='coerce')
df['funding_gap_days'] = (df['last_funding_at'] - df['first_funding_at']).dt.days.fillna(0)

# Drop all date/text columns that can't be used in model
cols_to_drop = ['permalink','name','homepage_url','category_list','state_code',
                'region','city','founded_at','founded_month','founded_quarter',
                'first_funding_at','last_funding_at']
df.drop([c for c in cols_to_drop if c in df.columns], axis=1, inplace=True)

df.head()

## 6. Outlier Handling

> **Bug Fix 3 (Critical):** The original code used row-dropping inside a loop over all columns. Each iteration dropped rows, shrinking the DataFrame, causing massive data loss and a mismatch between `numeric_cols` (computed once before the loop) and the remaining columns. Fixed by **clipping** instead of dropping rows, and excluding the target column.

In [ ]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

# Exclude target from outlier treatment
if 'funding_total_usd' in numeric_cols:
    numeric_cols.remove('funding_total_usd')

# FIX 4: Use clip() instead of row-dropping - preserves dataset size
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print("Shape after outlier clipping:", df.shape)

## 7. Log Transform Target + Skewness Fix

In [ ]:
# Log transform target variable
df['log_funding'] = np.log1p(df['funding_total_usd'])

# Fix skewness in features
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
skew_values  = df[numeric_cols].skew()
high_skew    = skew_values[abs(skew_values) > 1]

print("Highly skewed columns:")
print(high_skew)

for col in high_skew.index:
    if col not in ['funding_total_usd', 'log_funding']:
        if (df[col] >= 0).all():
            df[col] = np.log1p(df[col])

print("\nSkewness after fix:")
print(df[numeric_cols].skew())

## 8. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['country_code'].value_counts().head(10).plot(kind='bar', ax=axes[0], title='Top 10 Countries')
sns.histplot(df['funding_rounds'], bins=30, ax=axes[1])
axes[1].set_title('Distribution of Funding Rounds')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
corr_matrix  = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 9. Encode Categoricals

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns
print("Categorical columns:", cat_cols.tolist())
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

## 10. Feature Selection

> **Bug Fix 4:** Original feature list was incomplete (missing round_D through round_H and the new `funding_gap_days`). Fixed to include all relevant columns.

In [ ]:
# FIX 5: Complete feature list including all rounds and funding_gap_days
features = ['funding_rounds','seed','venture','angel',
            'debt_financing','private_equity',
            'round_A','round_B','round_C','round_D',
            'round_E','round_F','round_G','round_H',
            'funding_gap_days','founded_year']

# Keep only features that exist in dataframe
features = [f for f in features if f in df.columns]
print("Features used:", features)

X = df[features].fillna(0)
y = df['log_funding']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 11. Model Training

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# FIX 6: Added max_depth to prevent Decision Tree from overfitting perfectly
dt = DecisionTreeRegressor(random_state=42, max_depth=8)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

rf = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

## 12. Evaluation

In [ ]:
comparison = pd.DataFrame({
    'Model':    ['Linear Regression', 'Random Forest', 'Decision Tree'],
    'R2 Score': [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_rf), r2_score(y_test, y_pred_dt)],
    'MAE':      [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_rf), mean_absolute_error(y_test, y_pred_dt)],
    'MSE':      [mean_squared_error(y_test, y_pred_lr),  mean_squared_error(y_test, y_pred_rf),  mean_squared_error(y_test, y_pred_dt)]
})
print(comparison)

comparison.set_index('Model')['R2 Score'].plot(kind='bar', color=['steelblue','forestgreen','coral'])
plt.title("Model Comparison — R² Score")
plt.ylabel("R² Score")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 13. Save Best Model (Random Forest)

In [ ]:
# FIX 7: Save the best model (RF), not Decision Tree
# Original saved dt as model.pkl but compared RF as best
joblib.dump(rf, 'model.pkl')
joblib.dump(features, 'columns.pkl')
print("✅ model.pkl and columns.pkl saved successfully!")